<a href="https://colab.research.google.com/github/CharlizeCo/house_prices_csc481_project/blob/main/house_price_prediction_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

NOTE: WE CAN NOT WORK ON THE COLAB AT THE SAME TIME BECAUSE CHANGES MADE BY ONE USER MIGHT BE OVERWRITE THE OTHER'S CHANGES

(it doesn't update in real time unlike google docs)

###Requirements

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
#from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

In [2]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE : {mae:.2f}")
    print(f"R^2 : {r2:.3f}")

###Dataset

In [22]:
house_url = "https://raw.githubusercontent.com/CharlizeCo/house_prices_csc481_project/refs/heads/main/House%20Price%20Prediction%20Dataset.csv"
house_df = pd.read_csv(house_url)
house_df.head(20)

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056
5,6,3944,1,2,1,1906,Urban,Poor,No,93262
6,7,3671,1,1,2,1948,Rural,Poor,Yes,448722
7,8,3419,2,4,1,1925,Suburban,Good,Yes,594893
8,9,630,2,2,1,1932,Rural,Poor,Yes,652878
9,10,2185,3,3,1,2000,Downtown,Poor,No,340375


In [23]:
list(house_df.columns)

['Id',
 'Area',
 'Bedrooms',
 'Bathrooms',
 'Floors',
 'YearBuilt',
 'Location',
 'Condition',
 'Garage',
 'Price']

In [24]:
print(house_df.describe())

                Id         Area     Bedrooms   Bathrooms       Floors  \
count  2000.000000  2000.000000  2000.000000  2000.00000  2000.000000   
mean   1000.500000  2786.209500     3.003500     2.55250     1.993500   
std     577.494589  1295.146799     1.424606     1.10899     0.809188   
min       1.000000   501.000000     1.000000     1.00000     1.000000   
25%     500.750000  1653.000000     2.000000     2.00000     1.000000   
50%    1000.500000  2833.000000     3.000000     3.00000     2.000000   
75%    1500.250000  3887.500000     4.000000     4.00000     3.000000   
max    2000.000000  4999.000000     5.000000     4.00000     3.000000   

         YearBuilt          Price  
count  2000.000000    2000.000000  
mean   1961.446000  537676.855000  
std      35.926695  276428.845719  
min    1900.000000   50005.000000  
25%    1930.000000  300098.000000  
50%    1961.000000  539254.000000  
75%    1993.000000  780086.000000  
max    2023.000000  999656.000000  


In [25]:
# set up features and target
features = [
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "YearBuilt",
]

target = "Price"

X = house_df[features]
y = house_df[target]

# check for missing values
print(house_df.isnull().sum())

# remove duplicates if needed
df_fitbit = house_df.drop_duplicates()

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# view the split data
#print(f"{X_train}\n{X_test}\n{y_train}\n{y_test}")

# for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Id           0
Area         0
Bedrooms     0
Bathrooms    0
Floors       0
YearBuilt    0
Location     0
Condition    0
Garage       0
Price        0
dtype: int64


###MODEL SECTION (inital results)
---

**First Model**

In [26]:
# first model (Linear Regression)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

**Second Model**

In [27]:
# second model (Random Forest)
rf = RandomForestRegressor(n_estimators=100) #model
rf.fit(X_train, y_train) #fit data
rf_pred = rf.predict(X_test) #predict data

**Third Model**

In [28]:
# third model (XGBoost)
xgb = GradientBoostingRegressor(n_estimators=100)
xgb.fit(X_train, y_train) #fit data
xgb_pred = xgb.predict(X_test) #predict data

**Evaluate Models**

In [31]:
evaluate_model("Linear Regression", y_test, lr_pred)
print("---")
evaluate_model("Random Forest Regression", y_test, rf_pred)
print("---")
evaluate_model("Gradient Boost Regression", y_test, xgb_pred)


Linear Regression
RMSE: 280695.07
MAE : 244587.20
R^2 : -0.013
---

Random Forest Regression
RMSE: 293039.21
MAE : 254091.54
R^2 : -0.104
---

Gradient Boost Regression
RMSE: 284119.29
MAE : 246088.83
R^2 : -0.038


###IMPROVE MODELS

Identify important features and combine features in accordance

In [10]:
from sklearn.ensemble import RandomForestClassifier
# random forest classification can help us find the feature importances by
# finding which model contributes the most to the model's accuracy
clf = RandomForestClassifier(n_estimators=300)

clf.fit(X_train, y_train) # fit data
importances = clf.feature_importances_
# store feature importance into a dataset w/ feature
feature_importance = pd.DataFrame({'Feature': features, "Importance": importances}).sort_values('Importance', ascending = False) # sort
print(feature_importance)

     Feature  Importance
0       Area    0.407756
4  YearBuilt    0.361842
1   Bedrooms    0.090293
2  Bathrooms    0.078593
3     Floors    0.061516


In [32]:
# time based feature combination
current_year = 2026
house_df['Age'] = current_year - house_df['YearBuilt']

# house feature ratios
house_df['Price_per_Sqft'] = house_df['Price'] / house_df['Area']
house_df['Area_per_Bedroom'] = house_df['Area'] / (house_df['Bedrooms'] + 1)
house_df['Area_per_Bathroom'] = house_df['Area'] / (house_df['Bathrooms'] + 1)
house_df['Room_to_Area_Ratio'] = (house_df['Bedrooms'] + house_df['Bathrooms']) / house_df['Area']
house_df['Luxury_Index'] = (house_df['Bathrooms'] / house_df['Bedrooms']) * house_df['Floors']

# category scores
condition_scores = {'Excellent': 4, 'Good': 3, 'Fair': 2, 'Poor': 1}
location_scores = {'Downtown': 4, 'Urban': 3, 'Suburban': 2, 'Rural': 1}

house_df['Condition_Score'] = house_df['Condition'].map(condition_scores)
house_df['Location_Score'] = house_df['Location'].map(location_scores)
house_df['Garage_Binary'] = (house_df['Garage'] == 'Yes').astype(int)
house_df.head(11)

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price,Age,Price_per_Sqft,Area_per_Bedroom,Area_per_Bathroom,Room_to_Area_Ratio,Luxury_Index,Condition_Score,Location_Score,Garage_Binary
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919,56,110.234559,226.666667,272.000000,0.006618,2.4,4,4,0
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998,68,99.484551,712.000000,854.400000,0.002107,2.4,4,4,0
2,3,3592,2,2,3,1938,Downtown,Good,No,266746,88,74.261136,1197.333333,1197.333333,0.001114,3.0,3,4,0
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020,124,252.608696,193.200000,322.000000,0.006211,1.0,2,2,1
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056,51,129.122209,2463.000000,985.200000,0.001015,8.0,2,4,1
5,6,3944,1,2,1,1906,Urban,Poor,No,93262,120,23.646552,1972.000000,1314.666667,0.000761,2.0,1,3,0
6,7,3671,1,1,2,1948,Rural,Poor,Yes,448722,78,122.234269,1835.500000,1835.500000,0.000545,2.0,1,1,1
7,8,3419,2,4,1,1925,Suburban,Good,Yes,594893,101,173.996198,1139.666667,683.800000,0.001755,2.0,3,2,1
8,9,630,2,2,1,1932,Rural,Poor,Yes,652878,94,1036.314286,210.000000,210.000000,0.006349,1.0,1,1,1
9,10,2185,3,3,1,2000,Downtown,Poor,No,340375,26,155.778032,546.250000,546.250000,0.002746,1.0,1,4,0


In [38]:
# set up features and target
features = [
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "YearBuilt",
    "Age",
    "Price_per_Sqft",
    "Area_per_Bedroom",
    "Area_per_Bathroom",
    "Room_to_Area_Ratio",
    "Luxury_Index",
    "Price_per_Sqft",
    "Condition_Score",
    "Location_Score",
    "Garage_Binary"
]

target = "Price"

X = house_df[features]
y = house_df[target]

# split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# view the split data
#print(f"{X_train}\n{X_test}\n{y_train}\n{y_test}")

###MODELS

In [39]:
# first model (Linear Regression)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

In [40]:
# second model (Random Forest)
rf = RandomForestRegressor(n_estimators=100) #model
rf.fit(X_train, y_train) #fit data
rf_pred = rf.predict(X_test) #predict data

In [41]:
# third model (XGBoost)
xgb = GradientBoostingRegressor(n_estimators=100)
xgb.fit(X_train, y_train) #fit data
xgb_pred = xgb.predict(X_test) #predict data

**Evaluate**

In [42]:
evaluate_model("Linear Regression", y_test, lr_pred)
print("---")
evaluate_model("Random Forest Regression", y_test, rf_pred)
print("---")
evaluate_model("Gradient Boost Regression", y_test, xgb_pred)


Linear Regression
RMSE: 172389.92
MAE : 135366.86
R^2 : 0.618
---

Random Forest Regression
RMSE: 22040.58
MAE : 13713.04
R^2 : 0.994
---

Gradient Boost Regression
RMSE: 30430.94
MAE : 22327.10
R^2 : 0.988


**Grid Search CV**